# 02 · Baselines on RICH (development cohort only)
Penalised Cox, gradient-boosted Cox, **monotone-constrained** gradient-boosted Cox, and an unconstrained MLP, for each predictor tier.
Repeated stratified CV inside RICH. NAGALA is not touched.

In [ ]:
import os, sys, pathlib, warnings
import numpy as np, pandas as pd, matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
sys.path.insert(0, ".")
import ckinn_lib as L

DATA = pathlib.Path(os.environ.get("CKINN_DATA", "data"))        # harmonized CSVs from notebook 01
OUT = pathlib.Path("results"); OUT.mkdir(exist_ok=True)
FIG = pathlib.Path("figs"); FIG.mkdir(exist_ok=True)
FAST = os.environ.get("CKINN_FAST", "0") == "1"                    # FAST=1: small subsample, code check only
SEED = 0
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

In [ ]:
rich, nag = L.load_data(DATA)
if FAST:
    rich = rich.sample(40000, random_state=SEED)                   # development cohort only
print(f"RICH {len(rich):,} rows, {rich.event.sum():,} events | FAST={FAST}")

## 1. Configuration

In [ ]:
TIER_LIST = ["T2", "T3"] if FAST else ["T1", "T2", "T3", "T4"]
K_FOLDS, REPEATS = (2, 1) if FAST else (5, 5)
XGB_N = 100 if FAST else 300
EPOCHS = 6 if FAST else 40
def specs(cols):
    return {"Cox (ridge)": lambda: L.CoxLinear(cols),
            "XGB-Cox": lambda: L.XGBCox(cols, False, XGB_N, seed=SEED),
            "XGB-Cox monotone": lambda: L.XGBCox(cols, True, XGB_N, seed=SEED),
            "MLP": lambda: L.DeepCox(cols, lam=0.0, epochs=EPOCHS, seed=SEED)}

## 2. Run (tier x repeat x fold)

In [ ]:
rows = []
for tier in TIER_LIST:
    cols = L.TIERS[tier]
    for rep in range(REPEATS):
        for k, (tr, te) in enumerate(L.cv_folds(rich, K_FOLDS, seed=rep)):
            res, _ = L.run_models(specs(cols), tr, te)
            res.insert(0, "tier", tier); res.insert(1, "repeat", rep); res.insert(2, "fold", k)
            rows.append(res)
        print(f"{tier} repeat {rep} done")
cv = pd.concat(rows, ignore_index=True)
cv.to_csv(OUT / "02_baselines_rich_cv.csv", index=False)

## 3. Summary (mean ± SD across folds)

In [ ]:
metrics = ["cindex", "auc_3y", "scaled_brier", "oe", "cal_slope", "viol_rate"]
summ = cv.groupby(["tier", "model"])[metrics].agg(["mean", "std"]).round(3)
summ

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.8))
for ax, m, t in zip(axes, ["cindex", "scaled_brier", "viol_rate"], ["C-index", "Scaled Brier (3 y)", "Constraint-violation rate"]):
    cv.pivot_table(index="tier", columns="model", values=m).plot.bar(ax=ax, rot=0); ax.set_title(t); ax.legend(fontsize=7)
plt.tight_layout(); plt.savefig(FIG / "02_baselines.png"); plt.show()

**Reading the table.** Tier T3 contains FPG, which nearly determines the outcome, so a high C-index there is expected and is not evidence for CKINN. Compare tiers T1 and T2 before drawing any conclusion.